# QoolQit Exercises — Module 3
## Compilation and Execution

In Module 2 we assembled device-agnostic `QuantumProgram`s. This module makes
them run: we **compile** programs to devices with real physical constraints,
and **execute** them on an emulator. Along the way you will run your first
genuinely quantum experiments: a **π-pulse** and the **Rydberg blockade**.

### In this module you will learn
- The built-in devices (`MockDevice`, `AnalogDevice`, ...) and their
  constraints
- How `compile_to` adapts a dimensionless program to hardware limits
- How to run a compiled program on the `LocalEmulator` and read out the
  measured bitstrings
- What a `CompilationError` means and how to reason about it


> **How to use this notebook.** 
> - Cells marked **✏️ Exercise** contain gaps
> indicated by `...` or `# TODO` — replace them with working code following
> the instructions. 
> - Cells marked **✅ Check** verify your answer: run them
> after completing the exercise. Everything else is provided and runs as-is.
> A separate **solution notebook** will be published.
>
> **API note:** we use qoolqit version 1.4

## 1. Devices

A `Device` bundles the physical constraints of a machine: maximum amplitude,
maximum sequence duration, minimum atom spacing, maximum radial distance.
QoolQit ships default devices you can use offline:

- **`MockDevice`** — a *virtual* device with (almost) no constraints, for
  unconstrained prototyping;
- **`AnalogDevice`** — a *realistic* analog device;
- **`AnalogDeviceWithDMM`**, **`DigitalAnalogDevice`** — variants with extra
  capabilities.

Real remote devices (e.g. Pasqal's FRESNEL) can be fetched with
`Device.from_connection(connection=PasqalCloud(), name="FRESNEL")` — same
interface, specs downloaded from the cloud.

### ✏️ Exercise 3.1 — Meet the devices

1. Call `available_default_devices()` (imported from `qoolqit`) to list the
   built-in devices and their constraints.
2. Instantiate `device = AnalogDevice()` and print it.
3. Read the printout and note down (mentally or in a comment): its
   **max_duration**, **max_amplitude** and **max_radial_distance**. Compare
   with `MockDevice()` — what is different?

In [ ]:
from qoolqit import MockDevice

# TODO: list the default devices

# TODO: instantiate and print the realistic analog device
device = ...
print(device)

print(MockDevice())

## 2. Compilation: your first physical experiment, the π-pulse

Compilation translates the dimensionless program into a concrete pulse
sequence in physical units, rescaling amplitude, duration and atom spacing to
fit the device (by default with the `MAX_ENERGY` profile, which uses the
device's maximum capabilities while preserving the program's ratios).

**The experiment.** Drive a *single atom* with a constant amplitude
$\Omega$ and zero detuning. The atom oscillates between $|0\rangle$ and
$|1\rangle$ (*Rabi oscillation*), and is fully flipped to $|1\rangle$ when

$$
\Omega \cdot t = \pi \qquad \text{(a "π-pulse")}.
$$

### ✏️ Exercise 3.2 — Compile a π-pulse

1. Build a **single-atom** register at the origin.
2. Build a drive with a `ConstantWaveform` amplitude of value `1.0` and a
   duration realizing a π-pulse. No detuning needed.
3. Assemble the program, compile it to the `AnalogDevice` with
   `program.compile_to(device=..., profile="max_energy")`, check `is_compiled`, and draw the
   compiled sequence with `program.draw(compiled=True)`.

Note the units in the drawing: compilation has translated dimensionless time
and amplitude into nanoseconds and rad/µs.

In [ ]:
import numpy as np
from qoolqit import ConstantWaveform, Drive, QuantumProgram, Register

# TODO: single atom at the origin
register_1atom = ...

# TODO: constant pi-pulse: Omega = 1.0, duration such that Omega*t = pi
pi_pulse = ConstantWaveform(..., ...)
drive_pi = Drive(amplitude=pi_pulse)

# TODO: assemble, compile to the AnalogDevice, and draw compiled
program_pi = ...
program_pi.compile_to(device=..., profile="max_energy")
print("Compiled?", program_pi.is_compiled)
program_pi.draw(compiled=True)

In [ ]:
# ✅ Check
assert program_pi.is_compiled
assert abs(drive_pi.duration - np.pi) < 1e-9
print("π-pulse compiled!")

## 3. Execution on the LocalEmulator

Executing follows a simple, backend-independent workflow:

```
emulator = LocalEmulator()      # from qoolqit.execution
job      = emulator.run(program)
results  = job.results()
counts   = results.final_bitstrings   # a Counter of measured bitstrings
```

Remote backends (`RemoteEmulator`, `QPU`) expose exactly the same `run` /
`results` interface — only the construction differs (they need a cloud
connection).

### ✏️ Exercise 3.3 — Run the π-pulse

Run `program_pi` on a `LocalEmulator` and print the measured bitstring
counts. If your pulse is a true π-pulse, (almost) **all shots should return**
`'1'` — the atom is deterministically flipped.

In [ ]:
# TODO: run the program and get the bitstring counts
emulator = ...
job = ...
results = ...
counts = ...

print(counts)

In [ ]:
# ✅ Check — at least 95% of shots in '1'
total = sum(counts.values())
assert counts.get("1", 0) / total > 0.95, "Expected (almost) all shots in '1'"
print(f"π-pulse verified: {counts.get('1', 0)}/{total} shots measured '1'.")

## 4. A two-atom experiment: the Rydberg blockade

Now the same π-pulse on **two atoms**. The interaction $J = 1/r^6$ changes
everything:

- **Far apart** ($J \ll \Omega$): the atoms don't feel each other and are
  *independently* flipped → you measure `'11'`.
- **Close together** ($J \gg \Omega$): exciting *both* atoms costs a huge
  interaction energy, so the doubly-excited state is **blockaded** → `'11'`
  is (almost) never measured. This *Rydberg blockade* is the fundamental
  mechanism behind unit-disk connectivity (Module 1) and neutral-atom
  entanglement.

### ✏️ Exercise 3.4 — Observe the blockade

1. `register_far`: two atoms at distance **3.0** (so $J = 1/3^6 \approx
   0.0014 \ll 1$). Apply the same π-pulse drive, compile to the
   `AnalogDevice`, run, and print the counts.
2. `register_close`: two atoms at distance **0.7** (so $J = 1/0.7^6 \approx
   8.5 \gg 1$). Same pulse, compile, run, print.
3. Compare the frequency of `'11'` in the two cases. Blockade in action!

In [ ]:
# TODO 1: two distant atoms -> independent flips
register_far = Register.from_coordinates([...])
program_far = QuantumProgram(register_far, Drive(amplitude=pi_pulse))
program_far.compile_to(device=device, profile="max_energy")
counts_far = ...
print("far  (r=3.0):", counts_far)

# TODO 2: two close atoms -> blockade
register_close = Register.from_coordinates([...])
program_close = ...
counts_close = ...
print("close(r=0.7):", counts_close)

In [ ]:
# ✅ Check
tot_far = sum(counts_far.values())
tot_close = sum(counts_close.values())
assert counts_far.get("11", 0) / tot_far > 0.9, (
    "Far atoms should (almost) always give '11'"
)
assert counts_close.get("11", 0) / tot_close < 0.05, (
    "Close atoms should (almost) never give '11'"
)
print("Rydberg blockade observed!")

## 5. When compilation fails: `CompilationError`

Compilation rescales amplitude, duration and spacing *together* to fit the
device. Sometimes no consistent rescaling exists — e.g. bringing a large
amplitude down to the device maximum stretches the duration beyond the device
limit. QoolQit then raises a **`CompilationError`** with a message explaining
which constraint broke.

### ✏️ Exercise 3.5 — Trigger and read a CompilationError

1. Build `program_bad`: the **close** two-atom register with a
   `ConstantWaveform(400, 1.0)` amplitude — a very long pulse.
2. Try to compile it to the `AnalogDevice` inside a
   `try/except CompilationError` block (import it from `qoolqit.exceptions`)
   and print the message. Read it: which device limit was violated?
3. Now compile the **same** program to a `MockDevice()` — it succeeds! Why is
   that both useful and dangerous?

In [ ]:
from qoolqit.exceptions import CompilationError

# TODO: an over-long program
program_bad = QuantumProgram(
    register_close, Drive(amplitude=ConstantWaveform(..., ...))
)

try:
    ...
except CompilationError as err:
    print("CompilationError:", err)

# TODO: same program on the unconstrained MockDevice
print("Compiled on MockDevice?", program_bad.is_compiled)